# forecast_validate.ipynb

Replicates `forecast_model.py` + `validate_model.py` end-to-end:
retrains the skforecast multi-series recursive forecaster on the
monthly Chicago-crime panel, runs rolling 4-step-ahead backtests on
the validate/forecast splits, and renders the model-validation
dashboard to `docs/forecast_dashboard.html`.


## Imports

In [1]:
import os
from math import cos, radians

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from skforecast.recursive import ForecasterRecursiveMultiSeries
from skforecast.utils import load_forecaster, save_forecaster
from xgboost import XGBRegressor

/Users/joehahn/Library/CloudStorage/Dropbox/prog/claude/chicago_crime_forecast/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load crimes_monthly.csv and split by TTV

In [2]:
df_monthly = pd.read_csv('data/crimes_monthly.csv', parse_dates=['date'])

keep_cols = [
    'date', 'year', 'month', 'ward', 'primary_type',
    'delta_count', 'count_0', 'count_1', 'count_2', 'count_3', 'count_4',
]
df_train    = df_monthly.loc[df_monthly['TTV'] == 'train',    keep_cols].copy()
df_validate = df_monthly.loc[df_monthly['TTV'] == 'validate', keep_cols].copy()
df_forecast = df_monthly.loc[df_monthly['TTV'] == 'forecast', keep_cols].copy()

print(f'df_train    has {len(df_train):,} records')
print(f'df_validate has {len(df_validate):,} records')
print(f'df_forecast has {len(df_forecast):,} records')
df_train.sample(n=5, random_state=0)

df_train    has 32,000 records
df_validate has 14,000 records
df_forecast has 1,000 records


,date,year,month,ward,primary_type,delta_count,count_0,count_1,count_2,count_3,count_4
46994,2022-07-01,2022,7,49,WEAPONS VIOLATION,-4.0,4,4.0,9.0,5.0,2.0
5258,2024-07-01,2024,7,6,INTERFERENCE WITH PUBLIC OFFICER,-1.0,2,4.0,2.0,4.0,5.0
18539,2023-04-01,2023,4,20,CRIMINAL TRESPASS,4.0,11,7.0,7.0,14.0,16.0
38887,2022-12-01,2022,12,41,INTIMIDATION,0.0,0,0.0,0.0,0.0,1.0
47822,2023-07-01,2023,7,50,ROBBERY,3.0,14,13.0,4.0,5.0,7.0


## Reshape training data for skforecast

In [3]:
df_train['series_id'] = (
    'w' + df_train['ward'].astype(int).astype(str).str.zfill(2)
    + '_' + df_train['primary_type'].str.replace(' ', '_')
)

series_wide = (
    df_train.pivot_table(index='date', columns='series_id', values='count_0', aggfunc='first')
    .sort_index()
    .asfreq('MS')
)

exog = (
    df_train[['date', 'year', 'month']]
    .drop_duplicates()
    .set_index('date')
    .sort_index()
    .asfreq('MS')
)

print('series_wide shape:', series_wide.shape)
print('exog shape:', exog.shape)

series_wide shape: (32, 1000)
exog shape: (32, 2)


## Train the forecaster

In [4]:
regressor = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

forecaster = ForecasterRecursiveMultiSeries(
    regressor=regressor,
    lags=6,
    encoding='ordinal',
)
forecaster.fit(series=series_wide, exog=exog)

os.makedirs('models', exist_ok=True)
save_forecaster(forecaster, file_name='models/forecaster.joblib', verbose=False)
print('saved forecaster to models/forecaster.joblib')

╭────────────────────────────────── InputTypeWarning ──────────────────────────────────╮
│ Passing a DataFrame (either wide or long format) as `series` requires additional     │
│ internal transformations, which can increase computational time. It is recommended   │
│ to use a dictionary of pandas Series instead. For more details, see:                 │
│ https://skforecast.org/latest/user_guides/independent-multi-time-series-forecasting. │
│ html#input-data                                                                      │
│                                                                                      │
│ Category : skforecast.exceptions.InputTypeWarning                                    │
│ Location :                                                                           │
│ /Users/joehahn/Library/CloudStorage/Dropbox/prog/claude/chicago_crime_forecast/.venv │
│ /lib/python3.9/site-packages/skforecast/utils/utils.py:2350                          │
│ Suppress : warnings.simplefilter('ignore', category=InputTypeWarning)                │
╰──────────────────────────────────────────────────────────────────────────────────────╯

saved forecaster to models/forecaster.joblib


## Build the full wide panel of actuals

In [5]:
df_all = df_monthly.copy()
df_all['series_id'] = (
    'w' + df_all['ward'].astype(int).astype(str).str.zfill(2)
    + '_' + df_all['primary_type'].str.replace(' ', '_')
)
panel = (
    df_all.pivot_table(index='date', columns='series_id', values='count_0', aggfunc='first')
    .sort_index().asfreq('MS')
)

def exog_for_dates(dates):
    d = pd.DatetimeIndex(dates)
    return pd.DataFrame({'year': d.year, 'month': d.month}, index=d)

panel.shape

(48, 1000)

## Generate 4-step rolling forecasts at every validate/forecast date

In [6]:
pred_dates = sorted(pd.concat([df_validate['date'], df_forecast['date']]).unique())
print('origin dates:', len(pred_dates))

pred_store = {}
lags = 6
for t in pred_dates:
    t = pd.Timestamp(t)
    window = panel.loc[:t].tail(lags)
    future_dates = pd.date_range(start=t + pd.offsets.MonthBegin(1), periods=4, freq='MS')
    exog_future = exog_for_dates(future_dates)
    preds = forecaster.predict(
        steps=4, last_window=window, exog=exog_future, suppress_warnings=True
    )
    preds_wide = preds.reset_index().pivot_table(
        index=preds.index.name or 'index',
        columns='level', values='pred', aggfunc='first',
    )
    preds_wide.index = pd.to_datetime(preds_wide.index)
    preds_wide = preds_wide.sort_index()
    for series_id in preds_wide.columns:
        pred_store[(t, series_id)] = preds_wide[series_id].tolist()


def attach_predictions(df):
    df = df.copy()
    df['series_id'] = (
        'w' + df['ward'].astype(int).astype(str).str.zfill(2)
        + '_' + df['primary_type'].str.replace(' ', '_')
    )
    for n in (1, 2, 3, 4):
        col = f'count_{n}_pred'
        df[col] = [
            pred_store.get((pd.Timestamp(d), sid), [np.nan]*4)[n-1]
            for d, sid in zip(df['date'], df['series_id'])
        ]
    return df


df_validate = attach_predictions(df_validate)
df_forecast = attach_predictions(df_forecast)
df_validate[(df_validate['primary_type']=='THEFT') & (df_validate['ward']==27)]

origin dates: 15


,date,year,month,ward,primary_type,delta_count,count_0,count_1,count_2,count_3,count_4,series_id,count_1_pred,count_2_pred,count_3_pred,count_4_pred
25856,2025-01-01,2025,1,27,THEFT,-31.0,209,180.0,202.0,248.0,254.0,w27_THEFT,227.828140,225.407639,224.581543,238.348953
25857,2025-02-01,2025,2,27,THEFT,-29.0,180,202.0,248.0,254.0,268.0,w27_THEFT,203.458084,190.455276,195.138901,192.883163
25858,2025-03-01,2025,3,27,THEFT,22.0,202,248.0,254.0,268.0,264.0,w27_THEFT,190.455276,195.138901,192.883163,191.682266
25859,2025-04-01,2025,4,27,THEFT,46.0,248,254.0,268.0,264.0,237.0,w27_THEFT,223.671783,239.389664,265.851013,259.982697
25860,2025-05-01,2025,5,27,THEFT,6.0,254,268.0,264.0,237.0,223.0,w27_THEFT,239.389664,265.851013,259.982697,287.527954
25861,2025-06-01,2025,6,27,THEFT,14.0,268,264.0,237.0,223.0,271.0,w27_THEFT,265.851013,259.982697,287.527954,276.225861
25862,2025-07-01,2025,7,27,THEFT,-4.0,264,237.0,223.0,271.0,221.0,w27_THEFT,259.982697,287.527954,276.225861,252.366562
25863,2025-08-01,2025,8,27,THEFT,-27.0,237,223.0,271.0,221.0,217.0,w27_THEFT,287.527954,276.225861,252.366562,233.338333
25864,2025-09-01,2025,9,27,THEFT,-14.0,223,271.0,221.0,217.0,202.0,w27_THEFT,276.225861,252.366562,233.338333,219.088409
25865,2025-10-01,2025,10,27,THEFT,48.0,271,221.0,217.0,202.0,159.0,w27_THEFT,252.366562,233.338333,219.088409,222.838364


## Validation scores

In [7]:
rows = []
for n in (1, 2, 3, 4):
    sub = df_validate[[f'count_{n}', f'count_{n}_pred']].dropna()
    y_t, y_p = sub.iloc[:, 0].to_numpy(), sub.iloc[:, 1].to_numpy()
    rows.append({
        'horizon': f't+{n} months',
        'MAE':  f'{mean_absolute_error(y_t, y_p):.3f}',
        'RMSE': f'{np.sqrt(mean_squared_error(y_t, y_p)):.3f}',
        'R2':   f'{r2_score(y_t, y_p):.3f}',
        'n':    len(sub),
    })
scores_df = pd.DataFrame(rows)
scores_df

,horizon,MAE,RMSE,R2,n
0,t+1 months,3.654,6.573,0.955,14000
1,t+2 months,4.242,8.367,0.925,14000
2,t+3 months,4.383,8.766,0.918,13000
3,t+4 months,4.542,9.304,0.907,12000


## Feature importances

In [8]:
fi = forecaster.get_feature_importances().sort_values('importance', ascending=False).reset_index(drop=True)
fi_disp = fi.astype(str)
fi_disp['importance'] = fi_disp['importance'].str.slice(0, 5)
fi_disp

,feature,importance
0,lag_1,0.641
1,lag_2,0.288
2,lag_3,0.017
3,year,0.012
4,month,0.010
5,lag_4,0.008
6,lag_6,0.007
7,lag_5,0.007
8,_level_skforecast,0.006


## Dashboard

Running `validate_model.py` end-to-end rebuilds `docs/forecast_dashboard.html`.
This cell shells out to it so the notebook produces the same artifact.

In [9]:
import subprocess
res = subprocess.run(['python3', 'validate_model.py'], capture_output=True, text=True)
print(res.stdout[-2000:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-2000:])

.0    221.0  w27_THEFT    259.982697    287.527954    276.225861    252.366562
25863 2025-08-01  2025      8    27        THEFT        -27.0      237    223.0    271.0    221.0    217.0  w27_THEFT    287.527954    276.225861    252.366562    233.338333
25864 2025-09-01  2025      9    27        THEFT        -14.0      223    271.0    221.0    217.0    202.0  w27_THEFT    276.225861    252.366562    233.338333    219.088409
25865 2025-10-01  2025     10    27        THEFT         48.0      271    221.0    217.0    202.0    159.0  w27_THEFT    252.366562    233.338333    219.088409    222.838364
25866 2025-11-01  2025     11    27        THEFT        -50.0      221    217.0    202.0    159.0    197.0  w27_THEFT    233.338333    219.088409    222.838364    240.837006
25867 2025-12-01  2025     12    27        THEFT         -4.0      217    202.0    159.0    197.0     58.0  w27_THEFT    219.088409    222.838364    240.837006    224.581543
25868 2026-01-01  2026      1    27        THEFT   